# Étape 4 — Sécurisation et conformité RGPD

Ce notebook ajoute à l'API :
- **Authentification par clé API** (header `X-API-Key`) avec comparaison à temps constant
- **Logs pseudonymes** (score + longueur, jamais le texte brut)
- **Registre RGPD** : qui accède, quand, avec quel résultat
- **IAM** : gestion des identités et des accès sur GKE

Utilise le scorer et le prétraitement de la branche Enzo (`api/scorer.py`, `api/preprocessing.py`).

In [ ]:
%pip install -q fastapi uvicorn nest-asyncio python-multipart

## 1. Chargement du modèle

In [ ]:
import sys
from pathlib import Path

RACINE = Path("..") if Path("../models").exists() else Path(".")
API_DIR = RACINE / "api"
sys.path.insert(0, str(API_DIR))

# Scorer et preprocessing d'Enzo (source unique de vérité pour le prétraitement)
from scorer import load_scorer, LABELS
from preprocessing import preprocess

scorer = load_scorer()
print(f"Modèle chargé : {scorer.version}")

## 2. Application FastAPI sécurisée

In [ ]:
import os, csv, hashlib, secrets, time
from datetime import datetime, timezone
from fastapi import FastAPI, HTTPException, Security, Depends
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field

API_KEY = os.getenv("API_KEY", "cle-dev-changer-en-prod")
LOG_PATH = RACINE / "logs" / "rgpd_access.csv"
LOG_PATH.parent.mkdir(exist_ok=True)
SEUIL = 0.5

class Demande(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)

class Reponse(BaseModel):
    score: int
    labels: dict
    model_version: str
    preprocessed_text: str

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verifier_cle(cle: str = Security(api_key_header)):
    # secrets.compare_digest : comparaison à temps constant (protection timing-attack)
    if cle is None or not secrets.compare_digest(cle, API_KEY):
        raise HTTPException(status_code=401, detail="Clé API absente ou invalide")
    return cle

def log_rgpd(cle: str, longueur: int, score: int, duree_ms: float):
    """Enregistre uniquement les métadonnées — jamais le texte brut (conformité RGPD)."""
    cle_hash = hashlib.sha256(cle.encode()).hexdigest()[:12]
    ligne = {"timestamp": datetime.now(timezone.utc).isoformat(),
             "cle_hash": cle_hash, "longueur": longueur,
             "score": score, "duree_ms": round(duree_ms, 1)}
    ecrire_en_tete = not LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ligne.keys())
        if ecrire_en_tete:
            writer.writeheader()
        writer.writerow(ligne)

app = FastAPI(title="Digital Social Score API", version="2.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["GET", "POST"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "model_version": scorer.version}

@app.post("/score", response_model=Reponse, dependencies=[Depends(verifier_cle)])
def score(demande: Demande, cle: str = Security(api_key_header)):
    t0 = time.perf_counter()
    texte = preprocess(demande.text)       # anonymisation + nettoyage (Enzo)
    probas = scorer.predict([texte])[0]    # inférence DistilBERT

    # Score = moyenne des labels > seuil ; fallback sur toxic si aucun actif
    actifs = [probas[l] for l in LABELS if probas[l] >= SEUIL]
    score_final = (round(100 * sum(actifs) / len(actifs))
                   if actifs else round(100 * probas["toxic"]))

    duree_ms = 1000 * (time.perf_counter() - t0)
    log_rgpd(cle, len(demande.text), score_final, duree_ms)
    return Reponse(
        score=score_final,
        labels={label: round(probas[label], 4) for label in LABELS},
        model_version=scorer.version,
        preprocessed_text=texte,
    )

print(f"App définie. Route : POST /score | Seuil : {SEUIL} | Clé : {API_KEY[:8]}…")

## 3. Lancement de l'API

In [ ]:
import nest_asyncio, uvicorn, threading, socket

nest_asyncio.apply()

# Choisit un port libre (8001 ou 8002 si déjà pris)
def port_libre(prefere=8001):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return prefere if s.connect_ex(("localhost", prefere)) != 0 else 8002

PORT = port_libre()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

thread = threading.Thread(target=_run, daemon=True)
thread.start()

import time; time.sleep(1)
print(f"✓ API démarrée sur http://localhost:{PORT}")
print(f"  Clé API : {API_KEY[:8]}…")
print("  Ouvre frontend/index.html dans ton navigateur.")

In [ ]:
# ── Optionnel : tunnel ngrok pour accès public ──────────────────────
# Nécessite un compte gratuit sur https://dashboard.ngrok.com/get-started/your-authtoken
# from pyngrok import ngrok
# ngrok.set_auth_token("COLLE_TON_TOKEN_ICI")
# public_url = ngrok.connect(PORT)
# print(f"URL publique : {public_url}")

## 4. Tests de sécurité

In [ ]:
import requests

BASE = f"http://localhost:{PORT}"

# ── Test 1 : sans clé → doit renvoyer 401 ──────────────────────────
resp = requests.post(f"{BASE}/score", json={"text": "hello world"})
print(f"Sans clé        → {resp.status_code} (attendu 401)")

# ── Test 2 : mauvaise clé → doit renvoyer 401 ──────────────────────
resp = requests.post(f"{BASE}/score",
                     json={"text": "hello world"},
                     headers={"X-API-Key": "mauvaise-cle"})
print(f"Mauvaise clé    → {resp.status_code} (attendu 401)")

# ── Test 3 : bonne clé → doit renvoyer 200 ─────────────────────────
resp = requests.post(f"{BASE}/score",
                     json={"text": "You are an idiot!"},
                     headers={"X-API-Key": API_KEY})
print(f"Bonne clé       → {resp.status_code} | {resp.json()}")

# ── Test 4 : texte trop long → doit renvoyer 422 ───────────────────
resp = requests.post(f"{BASE}/score",
                     json={"text": "a" * 5001},
                     headers={"X-API-Key": API_KEY})
print(f"Texte trop long → {resp.status_code} (attendu 422)")

## 5. Lecture du registre RGPD

In [ ]:
import pandas as pd
from pathlib import Path

log_path = Path("../logs/rgpd_access.csv")
if log_path.exists():
    logs = pd.read_csv(log_path)
    print(f"{len(logs)} appels enregistrés")
    print(logs.tail(10).to_string(index=False))
else:
    print("Aucun log pour l'instant — faites au moins un appel /score.")

## 6. Synthèse sécurité / RGPD

| Mesure | Implémentation |
|---|---|
| Authentification | Clé API dans header `X-API-Key` |
| Comparaison sécurisée | `secrets.compare_digest` — protection contre les timing-attacks |
| Anonymisation | spaCy NER + regex avant toute inférence (code d'Enzo) |
| Logs pseudonymes | Hash SHA-256 (12 chars) de la clé, longueur texte, score — jamais le texte brut |
| HTTPS | Certificat TLS géré par GKE Ingress (sslip.io) + ManagedCertificate |
| Clé API en production | Variable d'environnement injectée via **Secret Kubernetes** (`dss-api-key`) |
| Utilisateur non-root | Dockerfile : `useradd appuser` + `USER 1000` |
| Rotation des clés | Mettre à jour le Secret Kubernetes + `kubectl rollout restart` |

### Architecture de déploiement (branche Enzo)

```
Client → HTTPS → GKE Ingress (Load Balancer)
                        ↓
              Kubernetes Service (ClusterIP)
                        ↓
           Deployment dss-api (2 réplicas minimum)
                        ↓
              Pod : uvicorn + DistilBERT
```

### Commandes de déploiement GKE

```bash
# 1. Construire et pousser l'image (Cloud Build)
PROJECT_ID=mon-projet bash api/deploy/deploy_gke.sh

# 2. Créer le secret Kubernetes (jamais dans le code)
kubectl create secret generic dss-api-key \
  --from-literal=API_KEY=ma-cle-longue-et-aleatoire

# 3. Vérifier le déploiement
kubectl rollout status deployment/dss-api
kubectl get pods
```

## 7. IAM — Gestion des identités et des accès

L'IAM (**Identity and Access Management**) contrôle **qui** peut faire **quoi** sur l'infrastructure GCP/GKE.

### Principe du moindre privilège
Chaque composant n'obtient que les droits strictement nécessaires :

| Composant | Rôle IAM | Raison |
|---|---|---|
| Pod API (`dss-api-sa`) | `secretmanager.secretAccessor` | Lire la clé API uniquement |
| Cloud Build | `storage.admin` | Pousser l'image dans Artifact Registry |
| Nœuds GKE | `artifactregistry.reader` | Télécharger l'image au démarrage |

### Workload Identity
Lie un Service Account Kubernetes à un Service Account Google Cloud → les pods obtiennent leurs droits **sans stocker de clé JSON** dans l'image.

La cellule ci-dessous affiche les commandes à exécuter une seule fois lors de la mise en production.

In [ ]:
import textwrap

iam_steps = {
    "1. Créer un Service Account dédié": """
gcloud iam service-accounts create dss-api-sa \\
  --display-name='DSS API Service Account' \\
  --project=$PROJECT_ID
""",
    "2. Donner accès aux secrets uniquement (Secret Manager)": """
gcloud projects add-iam-policy-binding $PROJECT_ID \\
  --member='serviceAccount:dss-api-sa@$PROJECT_ID.iam.gserviceaccount.com' \\
  --role='roles/secretmanager.secretAccessor'
""",
    "3. Lier le SA Kubernetes au SA Google Cloud (Workload Identity)": """
kubectl annotate serviceaccount default \\
  iam.gke.io/gcp-service-account=dss-api-sa@$PROJECT_ID.iam.gserviceaccount.com

gcloud iam service-accounts add-iam-policy-binding \\
  dss-api-sa@$PROJECT_ID.iam.gserviceaccount.com \\
  --role='roles/iam.workloadIdentityUser' \\
  --member='serviceAccount:$PROJECT_ID.svc.id.goog[default/default]'
""",
    "4. Vérifier les droits actifs": """
gcloud projects get-iam-policy $PROJECT_ID \\
  --flatten='bindings[].members' \\
  --filter='bindings.members:dss-api-sa' \\
  --format='table(bindings.role)'
""",
}

for titre, cmd in iam_steps.items():
    print(f"{'─'*60}")
    print(f"  {titre}")
    print(f"{'─'*60}")
    print(textwrap.indent(cmd.strip(), "  "))
    print()

print("=" * 60)
print("  Résumé des rôles accordés")
print("=" * 60)
roles = [
    ("dss-api-sa",    "roles/secretmanager.secretAccessor", "Lire la clé API"),
    ("Cloud Build SA","roles/storage.admin",                "Pousser l'image Docker"),
    ("GKE node SA",   "roles/artifactregistry.reader",      "Télécharger l'image"),
]
print(f"{'Service Account':<20} {'Rôle IAM':<38} {'Usage'}")
print("-" * 85)
for sa, role, usage in roles:
    print(f"{sa:<20} {role:<38} {usage}")